In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    DecimalType
)

##streaming schema

In [0]:
transaction_schema = StructType([
    StructField("txn_id", StringType(), False),
    StructField("account_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("merchant_id", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("txn_time", TimestampType(), True),
    StructField("txn_type", StringType(), False),
    StructField("amount", DecimalType(18, 2), True),
    StructField("currency", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("status", StringType(), False)
])

print("Streaming schema created")

Streaming schema created


In [0]:
stream_source = "/Volumes/workspace/bronze/banking_source/streaming_transactions"

dbutils.fs.mkdirs(stream_source)

print("Streaming source:", stream_source)

Streaming source: /Volumes/workspace/bronze/banking_source/streaming_transactions


In [0]:
display(
    dbutils.fs.ls(stream_source)
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/_committed_3453369257889604469,_committed_3453369257889604469,735,1787112664000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/_committed_7537756147749838037,_committed_7537756147749838037,735,1787379479000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/_committed_8412140487786763954,_committed_8412140487786763954,746,1787048345000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/_committed_9013200018338211976,_committed_9013200018338211976,735,1787111355000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/_committed_vacuum7220070188053974145,_committed_vacuum7220070188053974145,162,1787379480000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/_started_7537756147749838037,_started_7537756147749838037,0,1787379478000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00000-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-151-1-c000.csv,part-00000-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-151-1-c000.csv,295440,1787379479000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00001-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-152-1-c000.csv,part-00001-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-152-1-c000.csv,295433,1787379479000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv,part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv,295430,1787379479000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv,part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv,295355,1787379479000


##Create streaming Bronze table

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.streaming_transactions (
    txn_id STRING,
    account_id STRING,
    customer_id STRING,
    merchant_id STRING,
    device_id STRING,
    txn_time TIMESTAMP,
    txn_type STRING,
    amount DECIMAL(18,2),
    currency STRING,
    channel STRING,
    status STRING,
    _ingest_ts TIMESTAMP,
    _source_file STRING
)
USING DELTA
""")

print("Streaming Bronze table ready")

Streaming Bronze table ready


##Read streaming source

In [0]:
stream_df = (
    spark.readStream
    .schema(transaction_schema)
    .option("header", "true")
    .option("maxFilesPerTrigger", 1)
    .csv(stream_source)
)

print("Streaming DataFrame created")

Streaming DataFrame created


##Add ingestion metadata

In [0]:
stream_bronze_df = (
    stream_df
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

#Start the streaming query

In [0]:
streaming_query = (
    stream_bronze_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/bronze/banking_source/checkpoints/streaming_bronze"
    )
    .trigger(availableNow=True)
    .toTable("workspace.bronze.streaming_transactions")
)

print("AvailableNow streaming query started")

AvailableNow streaming query started


In [0]:
print(streaming_query.isActive)
print(streaming_query.status)

True
{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [0]:
display(
    spark.table("workspace.bronze.streaming_transactions")
)

txn_id,account_id,customer_id,merchant_id,device_id,txn_time,txn_type,amount,currency,channel,status,_ingest_ts,_source_file
STREAM_TXN_00005000,ACC076359,CUST072627,MER021684,DEV040342,2026-08-22T06:17:55.886Z,DEBIT,13361.47,INR,UPI,SUCCESS,2026-08-22T06:18:18.319Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv
STREAM_TXN_00005001,ACC072975,CUST085482,MER001320,DEV059392,2026-08-22T06:17:55.886Z,DEBIT,56807.04,INR,ATM,SUCCESS,2026-08-22T06:18:18.319Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv
STREAM_TXN_00005002,ACC034835,CUST064907,MER006713,DEV042115,2026-08-22T06:17:55.886Z,CREDIT,15717.43,INR,ATM,SUCCESS,2026-08-22T06:18:18.319Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv
STREAM_TXN_00005003,ACC011675,CUST021970,MER004682,DEV055375,2026-08-22T06:17:55.886Z,DEBIT,14845.96,INR,CARD,SUCCESS,2026-08-22T06:18:18.319Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv
STREAM_TXN_00005004,ACC132325,CUST015454,MER015322,DEV010292,2026-08-22T06:17:55.886Z,DEBIT,74681.38,INR,ATM,SUCCESS,2026-08-22T06:18:18.319Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv
STREAM_TXN_00005005,ACC079587,CUST094516,MER016647,DEV037390,2026-08-22T06:17:55.886Z,DEBIT,78528.03,INR,ATM,SUCCESS,2026-08-22T06:18:18.319Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv
STREAM_TXN_00005006,ACC100510,CUST048295,MER004656,DEV029417,2026-08-22T06:17:55.886Z,DEBIT,38277.93,INR,UPI,SUCCESS,2026-08-22T06:18:18.319Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv
STREAM_TXN_00005007,ACC143807,CUST015217,MER009752,DEV006149,2026-08-22T06:17:55.886Z,CREDIT,48290.54,INR,CARD,SUCCESS,2026-08-22T06:18:18.319Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv
STREAM_TXN_00005008,ACC128921,CUST065927,MER008595,DEV042037,2026-08-22T06:17:55.886Z,DEBIT,49741.81,INR,ATM,SUCCESS,2026-08-22T06:18:18.319Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv
STREAM_TXN_00005009,ACC141273,CUST065697,MER019565,DEV010726,2026-08-22T06:17:55.886Z,DEBIT,72826.28,INR,ATM,SUCCESS,2026-08-22T06:18:18.319Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-153-1-c000.csv


##Generate streaming transactions

In [0]:
from pyspark.sql import functions as F

streaming_transactions = (
    spark.range(10000)
    .withColumn(
        "txn_id",
        F.concat(
            F.lit("STREAM_TXN_"),
            F.lpad(F.col("id").cast("string"), 8, "0")
        )
    )
    .withColumn(
        "account_id",
        F.concat(
            F.lit("ACC"),
            F.lpad(
                (F.floor(F.rand() * 150000)).cast("int").cast("string"),
                6,
                "0"
            )
        )
    )
    .withColumn(
        "customer_id",
        F.concat(
            F.lit("CUST"),
            F.lpad(
                (F.floor(F.rand() * 100000)).cast("int").cast("string"),
                6,
                "0"
            )
        )
    )
    .withColumn(
        "merchant_id",
        F.concat(
            F.lit("MER"),
            F.lpad(
                (F.floor(F.rand() * 25000)).cast("int").cast("string"),
                6,
                "0"
            )
        )
    )
    .withColumn(
        "device_id",
        F.concat(
            F.lit("DEV"),
            F.lpad(
                (F.floor(F.rand() * 75000)).cast("int").cast("string"),
                6,
                "0"
            )
        )
    )
    .withColumn(
        "txn_time",
        F.current_timestamp()
    )
    .withColumn(
        "txn_type",
        F.when(F.rand() < 0.5, "DEBIT").otherwise("CREDIT")
    )
    .withColumn(
        "amount",
        F.round(
            F.rand() * 100000 + 10,
            2
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "currency",
        F.lit("INR")
    )
    .withColumn(
        "channel",
        F.when(F.rand() < 0.30, "UPI")
         .when(F.rand() < 0.55, "ATM")
         .when(F.rand() < 0.75, "CARD")
         .otherwise("NETBANKING")
    )
    .withColumn(
        "status",
        F.when(F.rand() < 0.95, "SUCCESS")
         .otherwise("FAILED")
    )
    .drop("id")
)

print(
    "Streaming transactions generated:",
    f"{streaming_transactions.count():,}"
)

Streaming transactions generated: 10,000


In [0]:
display(
    streaming_transactions.limit(20)
)

txn_id,account_id,customer_id,merchant_id,device_id,txn_time,txn_type,amount,currency,channel,status
STREAM_TXN_00000000,ACC035559,CUST050557,MER016971,DEV054380,2026-08-22T07:02:07.365Z,CREDIT,57689.16,INR,UPI,SUCCESS
STREAM_TXN_00000001,ACC012981,CUST074920,MER009963,DEV058476,2026-08-22T07:02:07.365Z,CREDIT,84282.78,INR,CARD,SUCCESS
STREAM_TXN_00000002,ACC016685,CUST098708,MER006775,DEV017543,2026-08-22T07:02:07.365Z,CREDIT,13712.44,INR,ATM,SUCCESS
STREAM_TXN_00000003,ACC052781,CUST062831,MER021661,DEV028809,2026-08-22T07:02:07.365Z,DEBIT,61462.30,INR,ATM,SUCCESS
STREAM_TXN_00000004,ACC083313,CUST062795,MER019338,DEV029086,2026-08-22T07:02:07.365Z,DEBIT,54708.22,INR,ATM,SUCCESS
STREAM_TXN_00000005,ACC091468,CUST018732,MER021716,DEV066577,2026-08-22T07:02:07.365Z,DEBIT,3859.36,INR,CARD,SUCCESS
STREAM_TXN_00000006,ACC129407,CUST093895,MER016951,DEV056950,2026-08-22T07:02:07.365Z,DEBIT,99301.62,INR,ATM,SUCCESS
STREAM_TXN_00000007,ACC134132,CUST037052,MER004532,DEV071388,2026-08-22T07:02:07.365Z,DEBIT,77379.40,INR,UPI,SUCCESS
STREAM_TXN_00000008,ACC056801,CUST025480,MER021419,DEV029791,2026-08-22T07:02:07.365Z,CREDIT,51295.94,INR,UPI,FAILED
STREAM_TXN_00000009,ACC094578,CUST079025,MER001699,DEV035350,2026-08-22T07:02:07.365Z,CREDIT,3045.60,INR,UPI,SUCCESS


##Write events into the streaming source

In [0]:
(
    streaming_transactions
    .coalesce(4)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(stream_source)
)

print("10,000 streaming events written to source")

10,000 streaming events written to source


In [0]:
display(
    dbutils.fs.ls(stream_source)
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/_committed_5860636347555782391,_committed_5860636347555782391,739,1787382129000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/_committed_7537756147749838037,_committed_7537756147749838037,735,1787379479000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/_started_5860636347555782391,_started_5860636347555782391,0,1787382129000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00000-tid-5860636347555782391-7663e56a-d88e-489d-ad5f-02dc4e885c61-3408-1-c000.csv,part-00000-tid-5860636347555782391-7663e56a-d88e-489d-ad5f-02dc4e885c61-3408-1-c000.csv,295353,1787382129000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00001-tid-5860636347555782391-7663e56a-d88e-489d-ad5f-02dc4e885c61-3409-1-c000.csv,part-00001-tid-5860636347555782391-7663e56a-d88e-489d-ad5f-02dc4e885c61-3409-1-c000.csv,295254,1787382129000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00002-tid-5860636347555782391-7663e56a-d88e-489d-ad5f-02dc4e885c61-3410-1-c000.csv,part-00002-tid-5860636347555782391-7663e56a-d88e-489d-ad5f-02dc4e885c61-3410-1-c000.csv,295503,1787382129000
dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-5860636347555782391-7663e56a-d88e-489d-ad5f-02dc4e885c61-3411-1-c000.csv,part-00003-tid-5860636347555782391-7663e56a-d88e-489d-ad5f-02dc4e885c61-3411-1-c000.csv,295411,1787382129000


##Run the AvailableNow stream

In [0]:
streaming_query = (
    stream_bronze_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/bronze/banking_source/checkpoints/streaming_bronze"
    )
    .trigger(availableNow=True)
    .toTable("workspace.bronze.streaming_transactions")
)

print("✅ Streaming ingestion started")

✅ Streaming ingestion started


##Verify the streaming Bronze table

In [0]:
stream_count = spark.table(
    "workspace.bronze.streaming_transactions"
).count()

print(
    "Streaming Bronze records:",
    f"{stream_count:,}"
)

Streaming Bronze records: 50,000


##Verify the actual records

In [0]:
display(
    spark.table(
        "workspace.bronze.streaming_transactions"
    )
    .orderBy(F.desc("_ingest_ts"))
    .limit(20)
)

txn_id,account_id,customer_id,merchant_id,device_id,txn_time,txn_type,amount,currency,channel,status,_ingest_ts,_source_file
STREAM_TXN_00007506,ACC035208,CUST076422,MER006032,DEV062577,2026-08-22T06:17:55.886Z,DEBIT,93210.38,INR,CARD,SUCCESS,2026-08-22T06:18:22.573Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv
STREAM_TXN_00007500,ACC148683,CUST023839,MER021599,DEV054713,2026-08-22T06:17:55.886Z,CREDIT,57526.92,INR,CARD,SUCCESS,2026-08-22T06:18:22.573Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv
STREAM_TXN_00007509,ACC044862,CUST077282,MER003505,DEV041697,2026-08-22T06:17:55.886Z,CREDIT,38926.40,INR,ATM,SUCCESS,2026-08-22T06:18:22.573Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv
STREAM_TXN_00007502,ACC085756,CUST069815,MER018511,DEV017413,2026-08-22T06:17:55.886Z,CREDIT,22106.83,INR,CARD,SUCCESS,2026-08-22T06:18:22.573Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv
STREAM_TXN_00007507,ACC115466,CUST023298,MER024437,DEV011453,2026-08-22T06:17:55.886Z,DEBIT,28444.64,INR,ATM,SUCCESS,2026-08-22T06:18:22.573Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv
STREAM_TXN_00007511,ACC032968,CUST000745,MER016758,DEV048434,2026-08-22T06:17:55.886Z,DEBIT,7450.36,INR,ATM,SUCCESS,2026-08-22T06:18:22.573Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv
STREAM_TXN_00007516,ACC070767,CUST052840,MER024237,DEV019664,2026-08-22T06:17:55.886Z,DEBIT,19626.91,INR,UPI,SUCCESS,2026-08-22T06:18:22.573Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv
STREAM_TXN_00007512,ACC148499,CUST065328,MER018927,DEV034619,2026-08-22T06:17:55.886Z,CREDIT,89433.34,INR,CARD,SUCCESS,2026-08-22T06:18:22.573Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv
STREAM_TXN_00007514,ACC102135,CUST025269,MER001729,DEV065793,2026-08-22T06:17:55.886Z,CREDIT,69433.78,INR,ATM,SUCCESS,2026-08-22T06:18:22.573Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv
STREAM_TXN_00007515,ACC045557,CUST001280,MER014804,DEV073535,2026-08-22T06:17:55.886Z,DEBIT,11824.91,INR,ATM,SUCCESS,2026-08-22T06:18:22.573Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_transactions/part-00003-tid-7537756147749838037-5960fab6-80ca-496a-886f-9fd24af5ce17-154-1-c000.csv


#Create Streaming Silver table

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.streaming_transactions (
    txn_id STRING,
    account_id STRING,
    customer_id STRING,
    merchant_id STRING,
    device_id STRING,
    txn_time TIMESTAMP,
    txn_type STRING,
    amount DECIMAL(18,2),
    currency STRING,
    channel STRING,
    status STRING,
    _ingest_ts TIMESTAMP,
    _source_file STRING
)
USING DELTA
""")

print("Streaming Silver table ready")

Streaming Silver table ready


##Create quarantine table

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.streaming_quarantine (
    txn_id STRING,
    account_id STRING,
    customer_id STRING,
    merchant_id STRING,
    device_id STRING,
    txn_time TIMESTAMP,
    txn_type STRING,
    amount DECIMAL(18,2),
    currency STRING,
    channel STRING,
    status STRING,
    quarantine_reason STRING,
    _ingest_ts TIMESTAMP,
    _source_file STRING
)
USING DELTA
""")

print("Streaming quarantine table ready")

Streaming quarantine table ready


##Read streaming Bronze

In [0]:
streaming_silver_source = (
    spark.readStream
    .table("workspace.bronze.streaming_transactions")
)

print("Streaming Bronze source connected")

Streaming Bronze source connected


##Add watermark + deduplication

In [0]:
streaming_clean = (
    streaming_silver_source
    .withWatermark("txn_time", "10 minutes")
    .dropDuplicates(["txn_id"])
)

print("Watermark and transaction deduplication configured")

Watermark and transaction deduplication configured


##Validate transactions

In [0]:
valid_streaming = (
    streaming_clean
    .filter(F.col("amount") > 0)
    .filter(F.col("currency").isin("INR", "USD", "EUR", "GBP"))
    .filter(F.col("txn_type").isin("DEBIT", "CREDIT"))
    .filter(F.col("status") == "SUCCESS")
)

##Write Streaming Silver

In [0]:
silver_stream_query = (
    valid_streaming
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/bronze/banking_source/checkpoints/streaming_silver"
    )
    .trigger(availableNow=True)
    .toTable("workspace.silver.streaming_transactions")
)

print("Streaming Silver processing started")

Streaming Silver processing started


##Check Silver count

In [0]:
streaming_silver_count = spark.table(
    "workspace.silver.streaming_transactions"
).count()

print(
    "Streaming Silver records:",
    f"{streaming_silver_count:,}"
)

Streaming Silver records: 46,702


In [0]:
display(
    spark.table(
        "workspace.silver.streaming_transactions"
    )
    .orderBy(F.desc("_ingest_ts"))
    .limit(20)
)

txn_id,account_id,customer_id,merchant_id,device_id,txn_time,txn_type,amount,currency,channel,status,_ingest_ts,_source_file
STREAM_TEST_00007758,ACC073357,CUST001285,MER012547,DEV074091,2026-08-22T06:18:18.825Z,CREDIT,39386.67,INR,UPI,SUCCESS,2026-08-22T06:19:14.059Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_test_batch/part-00003-tid-7976558511572245116-79af1b3f-83ee-4204-b48e-13e0463d528c-185-1-c000.csv
STREAM_TEST_00007733,ACC141382,CUST022848,MER013914,DEV047296,2026-08-22T06:18:18.825Z,CREDIT,32127.12,INR,UPI,SUCCESS,2026-08-22T06:19:14.059Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_test_batch/part-00003-tid-7976558511572245116-79af1b3f-83ee-4204-b48e-13e0463d528c-185-1-c000.csv
STREAM_TEST_00007747,ACC031303,CUST075008,MER024161,DEV053659,2026-08-22T06:18:18.825Z,CREDIT,89514.54,INR,UPI,SUCCESS,2026-08-22T06:19:14.059Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_test_batch/part-00003-tid-7976558511572245116-79af1b3f-83ee-4204-b48e-13e0463d528c-185-1-c000.csv
STREAM_TEST_00007754,ACC115843,CUST025360,MER019590,DEV074079,2026-08-22T06:18:18.825Z,DEBIT,58404.89,INR,UPI,SUCCESS,2026-08-22T06:19:14.059Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_test_batch/part-00003-tid-7976558511572245116-79af1b3f-83ee-4204-b48e-13e0463d528c-185-1-c000.csv
STREAM_TEST_00007717,ACC082636,CUST003770,MER000721,DEV041507,2026-08-22T06:18:18.825Z,DEBIT,71871.43,INR,UPI,SUCCESS,2026-08-22T06:19:14.059Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_test_batch/part-00003-tid-7976558511572245116-79af1b3f-83ee-4204-b48e-13e0463d528c-185-1-c000.csv
STREAM_TEST_00007732,ACC048549,CUST023069,MER001560,DEV058982,2026-08-22T06:18:18.825Z,DEBIT,11458.85,INR,UPI,SUCCESS,2026-08-22T06:19:14.059Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_test_batch/part-00003-tid-7976558511572245116-79af1b3f-83ee-4204-b48e-13e0463d528c-185-1-c000.csv
STREAM_TEST_00007701,ACC016690,CUST058090,MER014101,DEV005818,2026-08-22T06:18:18.825Z,CREDIT,12418.81,INR,UPI,SUCCESS,2026-08-22T06:19:14.059Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_test_batch/part-00003-tid-7976558511572245116-79af1b3f-83ee-4204-b48e-13e0463d528c-185-1-c000.csv
STREAM_TEST_00007745,ACC136198,CUST073839,MER008847,DEV009657,2026-08-22T06:18:18.825Z,CREDIT,91975.95,INR,UPI,SUCCESS,2026-08-22T06:19:14.059Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_test_batch/part-00003-tid-7976558511572245116-79af1b3f-83ee-4204-b48e-13e0463d528c-185-1-c000.csv
STREAM_TEST_00007698,ACC091590,CUST059928,MER022709,DEV057038,2026-08-22T06:18:18.825Z,DEBIT,37802.38,INR,UPI,SUCCESS,2026-08-22T06:19:14.059Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_test_batch/part-00003-tid-7976558511572245116-79af1b3f-83ee-4204-b48e-13e0463d528c-185-1-c000.csv
STREAM_TEST_00007693,ACC069941,CUST096539,MER013084,DEV012912,2026-08-22T06:18:18.825Z,DEBIT,36368.23,INR,UPI,SUCCESS,2026-08-22T06:19:14.059Z,dbfs:/Volumes/workspace/bronze/banking_source/streaming_test_batch/part-00003-tid-7976558511572245116-79af1b3f-83ee-4204-b48e-13e0463d528c-185-1-c000.csv


##quarantine reason logic

In [0]:
def process_streaming_batch(batch_df, batch_id):

    batch_df = batch_df.dropDuplicates(["txn_id"])

    valid_df = (
        batch_df
        .filter(F.col("amount") > 0)
        .filter(F.col("currency").isin("INR", "USD", "EUR", "GBP"))
        .filter(F.col("txn_type").isin("DEBIT", "CREDIT"))
        .filter(F.col("status") == "SUCCESS")
    )

    invalid_df = (
        batch_df
        .withColumn(
            "quarantine_reason",
            F.when(
                F.col("amount").isNull() | (F.col("amount") <= 0),
                F.lit("INVALID_AMOUNT")
            )
            .when(
                ~F.col("currency").isin(
                    "INR", "USD", "EUR", "GBP"
                ),
                F.lit("INVALID_CURRENCY")
            )
            .when(
                ~F.col("txn_type").isin(
                    "DEBIT", "CREDIT"
                ),
                F.lit("INVALID_TXN_TYPE")
            )
            .when(
                F.col("status") != "SUCCESS",
                F.lit("UNSUCCESSFUL_TRANSACTION")
            )
            .otherwise(
                F.lit("UNKNOWN")
            )
        )
        .filter(F.col("quarantine_reason") != "UNKNOWN")
    )

    (
        valid_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "workspace.silver.streaming_transactions"
        )
    )

    (
        invalid_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "workspace.silver.streaming_quarantine"
        )
    )

    print(
        f"Batch {batch_id} processed"
    )

##Generate test batch

In [0]:
test_streaming_batch = (
    spark.range(10000)
    .withColumn(
        "txn_id",
        F.concat(
            F.lit("STREAM_TEST_"),
            F.lpad(
                F.col("id").cast("string"),
                8,
                "0"
            )
        )
    )
    .withColumn(
        "account_id",
        F.concat(
            F.lit("ACC"),
            F.lpad(
                (F.floor(F.rand() * 150000))
                .cast("int")
                .cast("string"),
                6,
                "0"
            )
        )
    )
    .withColumn(
        "customer_id",
        F.concat(
            F.lit("CUST"),
            F.lpad(
                (F.floor(F.rand() * 100000))
                .cast("int")
                .cast("string"),
                6,
                "0"
            )
        )
    )
    .withColumn(
        "merchant_id",
        F.concat(
            F.lit("MER"),
            F.lpad(
                (F.floor(F.rand() * 25000))
                .cast("int")
                .cast("string"),
                6,
                "0"
            )
        )
    )
    .withColumn(
        "device_id",
        F.concat(
            F.lit("DEV"),
            F.lpad(
                (F.floor(F.rand() * 75000))
                .cast("int")
                .cast("string"),
                6,
                "0"
            )
        )
    )
    .withColumn(
        "txn_time",
        F.current_timestamp()
    )
    .withColumn(
        "txn_type",
        F.when(F.rand() < 0.5, "DEBIT")
         .otherwise("CREDIT")
    )
    .withColumn(
        "amount",
        F.round(
            F.rand() * 100000 + 10,
            2
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "currency",
        F.lit("INR")
    )
    .withColumn(
        "channel",
        F.lit("UPI")
    )
    .withColumn(
        "status",
        F.lit("SUCCESS")
    )
    .drop("id")
)

In [0]:
test_streaming_batch = (
    test_streaming_batch
    .withColumn(
        "amount",
        F.when(
            F.rand() < 0.02,
            F.lit(-500).cast("decimal(18,2)")
        ).otherwise(F.col("amount"))
    )
    .withColumn(
        "currency",
        F.when(
            F.rand() < 0.02,
            F.lit("XYZ")
        ).otherwise(F.col("currency"))
    )
    .withColumn(
        "status",
        F.when(
            F.rand() < 0.03,
            F.lit("FAILED")
        ).otherwise(F.col("status"))
    )
)

#Write test batch

In [0]:
quarantine_test_source = (
    "/Volumes/workspace/bronze/"
    "banking_source/streaming_test_batch"
)

dbutils.fs.mkdirs(quarantine_test_source)

True

In [0]:
(
    test_streaming_batch
    .coalesce(4)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(quarantine_test_source)
)

print("Test streaming batch created")

Test streaming batch created


##Read the Test Streaming Source

In [0]:
test_stream_df = (
    spark.readStream
    .schema(transaction_schema)
    .option("header", "true")
    .option("maxFilesPerTrigger", 1)
    .csv(quarantine_test_source)
)

print("Test streaming source connected")

Test streaming source connected


In [0]:
test_stream_df = (
    test_stream_df
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

print("Streaming metadata added")

Streaming metadata added


In [0]:
quarantine_stream_query = (
    test_stream_df
    .writeStream
    .foreachBatch(process_streaming_batch)
    .option(
        "checkpointLocation",
        "/Volumes/workspace/bronze/"
        "banking_source/checkpoints/"
        "streaming_quarantine_test"
    )
    .trigger(availableNow=True)
    .start()
)

print(" Quarantine streaming pipeline started")

 Quarantine streaming pipeline started


In [0]:
print(quarantine_stream_query.status)
print("Active:", quarantine_stream_query.isActive)

{'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
Active: True


##Check Silver

In [0]:
silver_stream_count = spark.sql("""
    SELECT COUNT(*)
    FROM workspace.silver.streaming_transactions
""").collect()[0][0]

print(
    "Total Streaming Silver records:",
    f"{silver_stream_count:,}"
)

Total Streaming Silver records: 46,702


##Check Quarantine

In [0]:
quarantine_count = spark.sql("""
    SELECT COUNT(*)
    FROM workspace.silver.streaming_quarantine
""").collect()[0][0]

print(
    "Quarantined records:",
    f"{quarantine_count:,}"
)

Quarantined records: 2,784


##Analyze quarantine reasons

In [0]:
display(
    spark.sql("""
        SELECT
            quarantine_reason,
            COUNT(*) AS record_count
        FROM workspace.silver.streaming_quarantine
        GROUP BY quarantine_reason
        ORDER BY record_count DESC
    """)
)

quarantine_reason,record_count
UNSUCCESSFUL_TRANSACTION,1122
INVALID_AMOUNT,840
INVALID_CURRENCY,822


##Inspect quarantined transactions

In [0]:
display(
    spark.sql("""
        SELECT
            txn_id,
            account_id,
            amount,
            currency,
            txn_type,
            status,
            quarantine_reason,
            _ingest_ts
        FROM workspace.silver.streaming_quarantine
        ORDER BY _ingest_ts DESC
        LIMIT 50
    """)
)

txn_id,account_id,amount,currency,txn_type,status,quarantine_reason,_ingest_ts
STREAM_TEST_00008014,ACC144458,71809.94,INR,CREDIT,FAILED,UNSUCCESSFUL_TRANSACTION,2026-08-22T06:19:14.059Z
STREAM_TEST_00009775,ACC010996,-500.00,INR,CREDIT,SUCCESS,INVALID_AMOUNT,2026-08-22T06:19:14.059Z
STREAM_TEST_00007882,ACC099465,-500.00,INR,DEBIT,SUCCESS,INVALID_AMOUNT,2026-08-22T06:19:14.059Z
STREAM_TEST_00007976,ACC032068,82246.93,INR,CREDIT,FAILED,UNSUCCESSFUL_TRANSACTION,2026-08-22T06:19:14.059Z
STREAM_TEST_00007568,ACC050626,-500.00,INR,CREDIT,SUCCESS,INVALID_AMOUNT,2026-08-22T06:19:14.059Z
STREAM_TEST_00009496,ACC006267,82002.65,INR,DEBIT,FAILED,UNSUCCESSFUL_TRANSACTION,2026-08-22T06:19:14.059Z
STREAM_TEST_00008673,ACC066362,-500.00,INR,DEBIT,SUCCESS,INVALID_AMOUNT,2026-08-22T06:19:14.059Z
STREAM_TEST_00007857,ACC050341,-500.00,INR,DEBIT,SUCCESS,INVALID_AMOUNT,2026-08-22T06:19:14.059Z
STREAM_TEST_00008917,ACC062396,43627.31,INR,DEBIT,FAILED,UNSUCCESSFUL_TRANSACTION,2026-08-22T06:19:14.059Z
STREAM_TEST_00007712,ACC122782,18559.33,XYZ,DEBIT,SUCCESS,INVALID_CURRENCY,2026-08-22T06:19:14.059Z


##Streaming Gold & Fraud Detection

###Create Gold fraud table

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.streaming_fraud_transactions (
    txn_id STRING,
    account_id STRING,
    customer_id STRING,
    merchant_id STRING,
    device_id STRING,
    txn_time TIMESTAMP,
    txn_type STRING,
    amount DECIMAL(18,2),
    currency STRING,
    channel STRING,
    status STRING,

    high_value_flag BOOLEAN,
    velocity_flag BOOLEAN,
    channel_risk_flag BOOLEAN,

    fraud_score INT,
    risk_level STRING,

    _ingest_ts TIMESTAMP
)
USING DELTA
""")

print("Gold fraud transaction table created")

Gold fraud transaction table created


###Create account-risk table

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.streaming_account_risk (
    account_id STRING,
    transaction_count BIGINT,
    total_transaction_amount DECIMAL(18,2),
    high_value_transactions BIGINT,
    suspicious_transactions BIGINT,
    fraud_score BIGINT,
    risk_level STRING,
    last_transaction_time TIMESTAMP
)
USING DELTA
""")

print("Account risk table created")

Account risk table created


###Create channel metrics table

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.streaming_channel_metrics (
    channel STRING,
    transaction_count BIGINT,
    total_amount DECIMAL(18,2),
    average_amount DECIMAL(18,2),
    high_value_count BIGINT,
    suspicious_count BIGINT,
    metric_time TIMESTAMP
)
USING DELTA
""")

print("Channel metrics table created")

Channel metrics table created


###Read Streaming Silver

In [0]:
gold_stream_source = (
    spark.readStream
    .table("workspace.silver.streaming_transactions")
)

print("Streaming Silver connected")

Streaming Silver connected


In [0]:
fraud_features = (
    gold_stream_source
    .withColumn(
        "high_value_flag",
        F.col("amount") >= F.lit(100000)
    )
)

###Velocity detection

In [0]:
velocity_df = (
    fraud_features
    .withWatermark("txn_time", "10 minutes")
    .groupBy(
        F.window(
            "txn_time",
            "10 minutes"
        ),
        "account_id"
    )
    .agg(
        F.count("*").alias("transaction_count")
    )
    .withColumn(
        "velocity_flag",
        F.col("transaction_count") >= 10
    )
)

##production-style fraud processor

In [0]:
def process_fraud_batch(batch_df, batch_id):

    if batch_df.isEmpty():
        return

    # Remove duplicate transaction IDs within the batch
    batch_df = batch_df.dropDuplicates(["txn_id"])

    # --------------------------------------------------
    # 1. HIGH VALUE DETECTION
    # --------------------------------------------------

    fraud_df = (
        batch_df
        .withColumn(
            "high_value_flag",
            F.col("amount") >= F.lit(100000)
        )
    )

    # --------------------------------------------------
    # 2. VELOCITY DETECTION
    # --------------------------------------------------

    velocity_df = (
        batch_df
        .groupBy("account_id")
        .agg(
            F.count("*").alias("txn_count")
        )
    )

    fraud_df = (
        fraud_df
        .join(
            velocity_df,
            on="account_id",
            how="left"
        )
        .withColumn(
            "velocity_flag",
            F.col("txn_count") >= 10
        )
    )

    # --------------------------------------------------
    # 3. CHANNEL BEHAVIOR
    # --------------------------------------------------

    fraud_df = (
        fraud_df
        .withColumn(
            "channel_risk_flag",
            (
                (F.col("channel") == "ATM") &
                (F.col("amount") >= 50000)
            )
        )
    )

    # --------------------------------------------------
    # 4. FRAUD SCORE
    # --------------------------------------------------

    fraud_df = (
        fraud_df
        .withColumn(
            "fraud_score",
            F.when(F.col("high_value_flag"), 40)
             .otherwise(0)
            +
            F.when(F.col("velocity_flag"), 35)
             .otherwise(0)
            +
            F.when(F.col("channel_risk_flag"), 25)
             .otherwise(0)
        )
    )

    # --------------------------------------------------
    # 5. RISK LEVEL
    # --------------------------------------------------

    fraud_df = (
        fraud_df
        .withColumn(
            "risk_level",
            F.when(
                F.col("fraud_score") >= 70,
                "HIGH"
            )
            .when(
                F.col("fraud_score") >= 40,
                "MEDIUM"
            )
            .otherwise("LOW")
        )
    )

    # --------------------------------------------------
    # 6. WRITE FRAUD GOLD
    # --------------------------------------------------

    fraud_output = fraud_df.select(
        "txn_id",
        "account_id",
        "customer_id",
        "merchant_id",
        "device_id",
        "txn_time",
        "txn_type",
        "amount",
        "currency",
        "channel",
        "status",
        "high_value_flag",
        "velocity_flag",
        "channel_risk_flag",
        "fraud_score",
        "risk_level",
        "_ingest_ts"
    )

    (
        fraud_output
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "workspace.gold.streaming_fraud_transactions"
        )
    )

    # --------------------------------------------------
    # 7. ACCOUNT RISK
    # --------------------------------------------------

    account_risk = (
        fraud_output
        .groupBy("account_id")
        .agg(
            F.count("*").alias("transaction_count"),

            F.sum("amount")
             .cast("decimal(18,2)")
             .alias("total_transaction_amount"),

            F.sum(
                F.when(
                    F.col("high_value_flag"),
                    1
                ).otherwise(0)
            ).alias("high_value_transactions"),

            F.sum(
                F.when(
                    F.col("fraud_score") > 0,
                    1
                ).otherwise(0)
            ).alias("suspicious_transactions"),

            F.sum("fraud_score")
             .alias("fraud_score"),

            F.max("txn_time")
             .alias("last_transaction_time")
        )
        .withColumn(
            "risk_level",
            F.when(
                F.col("fraud_score") >= 100,
                "HIGH"
            )
            .when(
                F.col("fraud_score") >= 40,
                "MEDIUM"
            )
            .otherwise("LOW")
        )
    )

    (
        account_risk
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "workspace.gold.streaming_account_risk"
        )
    )

    print(
        f"Fraud batch {batch_id} processed successfully"
    )

##Start the Fraud Streaming Pipeline

In [0]:
fraud_stream_query = (
    gold_stream_source
    .writeStream
    .foreachBatch(process_fraud_batch)
    .option(
        "checkpointLocation",
        "/Volumes/workspace/bronze/"
        "banking_source/checkpoints/"
        "streaming_fraud_gold"
    )
    .trigger(availableNow=True)
    .start()
)

print("Streaming fraud pipeline started")

Streaming fraud pipeline started


In [0]:
print("Active:", fraud_stream_query.isActive)
print(fraud_stream_query.status)

Active: True
{'message': 'Initializing StreamExecution', 'isDataAvailable': False, 'isTriggerActive': False}


In [0]:
fraud_count = spark.sql("""
    SELECT COUNT(*)
    FROM workspace.gold.streaming_fraud_transactions
""").collect()[0][0]

print(
    "Fraud Gold records:",
    f"{fraud_count:,}"
)

Fraud Gold records: 28,766


In [0]:
display(
    spark.sql("""
        SELECT
            risk_level,
            COUNT(*) AS transaction_count
        FROM workspace.gold.streaming_fraud_transactions
        GROUP BY risk_level
        ORDER BY transaction_count DESC
    """)
)

risk_level,transaction_count
LOW,28761
MEDIUM,5


In [0]:
display(
    spark.sql("""
        SELECT
            txn_id,
            account_id,
            amount,
            channel,
            high_value_flag,
            velocity_flag,
            channel_risk_flag,
            fraud_score,
            risk_level
        FROM workspace.gold.streaming_fraud_transactions
        WHERE fraud_score > 0
        ORDER BY fraud_score DESC, amount DESC
        LIMIT 50
    """)
)

txn_id,account_id,amount,channel,high_value_flag,velocity_flag,channel_risk_flag,fraud_score,risk_level
STREAM_TXN_00009151,ACC069669,100000.41,ATM,true,false,true,65,MEDIUM
STREAM_TXN_00006272,ACC104632,100005.79,CARD,true,false,false,40,MEDIUM
STREAM_TEST_00003719,ACC009460,100005.36,UPI,true,false,false,40,MEDIUM
STREAM_TEST_00005937,ACC086812,100004.78,UPI,true,false,false,40,MEDIUM
STREAM_TEST_00004651,ACC006266,100003.80,UPI,true,false,false,40,MEDIUM
STREAM_TXN_00008386,ACC143689,99985.89,ATM,false,false,true,25,LOW
STREAM_TXN_00003252,ACC078750,99959.38,ATM,false,false,true,25,LOW
STREAM_TXN_00009976,ACC000412,99952.51,ATM,false,false,true,25,LOW
STREAM_TXN_00000711,ACC078021,99880.87,ATM,false,false,true,25,LOW
STREAM_TXN_00002081,ACC024729,99835.94,ATM,false,false,true,25,LOW


##Channel Analysis

In [0]:
channel_metrics = spark.sql("""
    SELECT
        channel,
        COUNT(*) AS transaction_count,
        CAST(SUM(amount) AS DECIMAL(18,2)) AS total_amount,
        CAST(AVG(amount) AS DECIMAL(18,2)) AS average_amount,
        SUM(
            CASE
                WHEN high_value_flag = true
                THEN 1
                ELSE 0
            END
        ) AS high_value_count,
        SUM(
            CASE
                WHEN fraud_score > 0
                THEN 1
                ELSE 0
            END
        ) AS suspicious_count,
        CURRENT_TIMESTAMP() AS metric_time
    FROM workspace.gold.streaming_fraud_transactions
    GROUP BY channel
""")

display(channel_metrics)

channel,transaction_count,total_amount,average_amount,high_value_count,suspicious_count,metric_time
ATM,3600,182198722.33,50610.76,1,1843,2026-08-22T07:03:05.262Z
UPI,22128,1107888548.29,50067.27,3,3,2026-08-22T07:03:05.262Z
NETBANKING,776,39873148.53,51382.92,0,0,2026-08-22T07:03:05.262Z
CARD,2262,114512018.95,50624.23,1,1,2026-08-22T07:03:05.262Z


##Save channel metrics

In [0]:
(
    channel_metrics
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "workspace.gold.streaming_channel_metrics"
    )
)

print("Channel metrics saved")

Channel metrics saved


#Account Balance & Reconciliation

###Create enhanced balance table

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.account_balance_summary (
    account_id STRING,
    transaction_count BIGINT,
    credit_amount DECIMAL(18,2),
    debit_amount DECIMAL(18,2),
    current_balance DECIMAL(18,2),
    last_transaction_time TIMESTAMP,
    balance_status STRING,
    calculated_at TIMESTAMP
)
USING DELTA
""")

print("Account balance summary table created")

Account balance summary table created


###Calculate balances

In [0]:
balance_df = (
    spark.table("workspace.silver.transactions")
    .groupBy("account_id")
    .agg(
        F.count("*").alias("transaction_count"),

        F.sum(
            F.when(
                F.col("txn_type") == "CREDIT",
                F.col("amount")
            ).otherwise(F.lit(0))
        )
        .cast("decimal(18,2)")
        .alias("credit_amount"),

        F.sum(
            F.when(
                F.col("txn_type") == "DEBIT",
                F.col("amount")
            ).otherwise(F.lit(0))
        )
        .cast("decimal(18,2)")
        .alias("debit_amount"),

        F.max("txn_time").alias("last_transaction_time")
    )
)

###Calculate current balance

In [0]:
balance_df = (
    balance_df
    .withColumn(
        "current_balance",
        (
            F.col("credit_amount")
            -
            F.col("debit_amount")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "balance_status",
        F.when(
            F.col("current_balance") < 0,
            "NEGATIVE"
        )
        .otherwise("NORMAL")
    )
    .withColumn(
        "calculated_at",
        F.current_timestamp()
    )
)

display(balance_df.limit(20))

account_id,transaction_count,credit_amount,debit_amount,last_transaction_time,current_balance,balance_status,calculated_at
ACC001657,35,2070466.20,1807853.50,2025-12-30T06:23:07.000Z,262612.70,NORMAL,2026-08-22T07:03:11.784Z
ACC004138,35,1465683.79,1440100.84,2025-12-15T15:59:50.000Z,25582.95,NORMAL,2026-08-22T07:03:11.784Z
ACC008039,34,2024900.49,1896967.46,2025-12-08T06:41:29.000Z,127933.03,NORMAL,2026-08-22T07:03:11.784Z
ACC009626,34,1104022.96,1878327.81,2025-12-28T01:36:30.000Z,-774304.85,NEGATIVE,2026-08-22T07:03:11.784Z
ACC011881,34,1410792.56,965042.37,2025-12-28T16:38:20.000Z,445750.19,NORMAL,2026-08-22T07:03:11.784Z
ACC012677,34,1544795.69,2042689.49,2025-12-31T21:40:47.000Z,-497893.80,NEGATIVE,2026-08-22T07:03:11.784Z
ACC012913,34,2035001.87,2019931.47,2025-12-21T10:31:26.000Z,15070.40,NORMAL,2026-08-22T07:03:11.784Z
ACC014764,33,1677663.86,3128141.08,2025-12-26T08:02:37.000Z,-1450477.22,NEGATIVE,2026-08-22T07:03:11.784Z
ACC021808,35,1841364.19,1475336.06,2025-12-14T01:01:23.000Z,366028.13,NORMAL,2026-08-22T07:03:11.784Z
ACC023596,35,2461464.29,1787852.61,2025-12-31T16:30:22.000Z,673611.68,NORMAL,2026-08-22T07:03:11.784Z


###Save Gold balances

In [0]:
(
    balance_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold.account_balance_summary"
    )
)

print("Account balances calculated and saved")

Account balances calculated and saved


##Validate against existing Gold table

In [0]:
display(
    spark.sql("""
        SELECT
            account_id,
            current_balance,
            last_updated
        FROM workspace.gold.account_balances
        ORDER BY account_id
        LIMIT 20
    """)
)

account_id,current_balance,last_updated
ACC000001,-614476.53,2025-12-17T03:21:13.000Z
ACC000003,-296844.41,2025-12-28T23:10:55.000Z
ACC000004,-695565.57,2025-12-26T13:40:06.000Z
ACC000006,157143.53,2025-12-16T09:09:18.000Z
ACC000007,-744247.47,2025-12-31T17:15:40.000Z
ACC000008,-515219.16,2025-12-27T13:54:53.000Z
ACC000009,-64393.12,2025-12-31T03:22:31.000Z
ACC000010,-175271.91,2025-12-06T01:48:18.000Z
ACC000011,-79266.58,2025-12-25T08:55:47.000Z
ACC000012,1097995.24,2025-12-31T22:20:00.000Z


In [0]:
display(
    spark.sql("""
        SELECT
            account_id,
            current_balance,
            credit_amount,
            debit_amount,
            transaction_count,
            balance_status
        FROM workspace.gold.account_balance_summary
        ORDER BY account_id
        LIMIT 20
    """)
)

account_id,current_balance,credit_amount,debit_amount,transaction_count,balance_status
ACC000001,-614476.53,1635365.07,2249841.60,35,NEGATIVE
ACC000003,-296844.41,1609686.65,1906531.06,35,NEGATIVE
ACC000004,-695565.57,1243936.83,1939502.40,34,NEGATIVE
ACC000006,157143.53,1718753.46,1561609.93,37,NORMAL
ACC000007,-744247.47,432267.35,1176514.82,32,NEGATIVE
ACC000008,-515219.16,1086014.98,1601234.14,35,NEGATIVE
ACC000009,-64393.12,2357022.32,2421415.44,35,NEGATIVE
ACC000010,-175271.91,1717623.08,1892894.99,35,NEGATIVE
ACC000011,-79266.58,1935133.23,2014399.81,36,NEGATIVE
ACC000012,1097995.24,1770404.41,672409.17,33,NORMAL


##Balance Quality Checks

In [0]:
balance_quality = spark.sql("""
SELECT
    COUNT(*) AS total_accounts,

    SUM(
        CASE
            WHEN current_balance < 0
            THEN 1
            ELSE 0
        END
    ) AS negative_balance_accounts,

    SUM(
        CASE
            WHEN current_balance = 0
            THEN 1
            ELSE 0
        END
    ) AS zero_balance_accounts,

    SUM(
        CASE
            WHEN current_balance > 0
            THEN 1
            ELSE 0
        END
    ) AS positive_balance_accounts
FROM workspace.gold.account_balance_summary
""")

display(balance_quality)

total_accounts,negative_balance_accounts,zero_balance_accounts,positive_balance_accounts
135058,88208,0,46850


##Find high-risk accounts

In [0]:
display(
    spark.sql("""
        SELECT
            b.account_id,
            b.current_balance,
            b.transaction_count,
            b.balance_status,
            r.suspicious_transactions,
            r.fraud_score,
            r.risk_level
        FROM workspace.gold.account_balance_summary b
        LEFT JOIN workspace.gold.streaming_account_risk r
            ON b.account_id = r.account_id
        WHERE r.risk_level IN ('HIGH', 'MEDIUM')
        ORDER BY r.fraud_score DESC
        LIMIT 50
    """)
)

account_id,current_balance,transaction_count,balance_status,suspicious_transactions,fraud_score,risk_level
ACC069669,-26149.82,35,NEGATIVE,2,90,MEDIUM
ACC104632,405860.56,35,NORMAL,2,65,MEDIUM
ACC020493,304404.62,35,NORMAL,2,50,MEDIUM
ACC050547,-694577.29,37,NEGATIVE,2,50,MEDIUM
ACC103421,-163017.69,34,NEGATIVE,2,50,MEDIUM
ACC095342,-663544.28,33,NEGATIVE,2,50,MEDIUM
ACC043822,494904.35,36,NORMAL,2,50,MEDIUM
ACC086900,-83147.87,36,NEGATIVE,2,50,MEDIUM
ACC127064,170398.92,31,NORMAL,2,50,MEDIUM
ACC032404,-1530483.97,32,NEGATIVE,2,50,MEDIUM


##Reconciliation

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.balance_reconciliation (
    account_id STRING,
    calculated_balance DECIMAL(18,2),
    source_balance DECIMAL(18,2),
    balance_difference DECIMAL(18,2),
    reconciliation_status STRING,
    checked_at TIMESTAMP
)
USING DELTA
""")

print("Reconciliation table created")

Reconciliation table created


###Reconcile

In [0]:
reconciliation_df = spark.sql("""
SELECT
    n.account_id,

    n.current_balance AS calculated_balance,

    o.current_balance AS source_balance,

    CAST(
        n.current_balance - o.current_balance
        AS DECIMAL(18,2)
    ) AS balance_difference,

    CASE
        WHEN ABS(
            n.current_balance - o.current_balance
        ) < 0.01
        THEN 'RECONCILED'

        ELSE 'MISMATCH'
    END AS reconciliation_status,

    CURRENT_TIMESTAMP() AS checked_at

FROM workspace.gold.account_balance_summary n

INNER JOIN workspace.gold.account_balances o
    ON n.account_id = o.account_id
""")

display(reconciliation_df.limit(20))

account_id,calculated_balance,source_balance,balance_difference,reconciliation_status,checked_at
ACC001657,262612.70,262612.70,0.00,RECONCILED,2026-08-22T07:03:24.116Z
ACC004138,25582.95,25582.95,0.00,RECONCILED,2026-08-22T07:03:24.116Z
ACC008039,127933.03,127933.03,0.00,RECONCILED,2026-08-22T07:03:24.116Z
ACC009626,-774304.85,-774304.85,0.00,RECONCILED,2026-08-22T07:03:24.116Z
ACC011881,445750.19,445750.19,0.00,RECONCILED,2026-08-22T07:03:24.116Z
ACC012677,-497893.80,-497893.80,0.00,RECONCILED,2026-08-22T07:03:24.116Z
ACC012913,15070.40,15070.40,0.00,RECONCILED,2026-08-22T07:03:24.116Z
ACC014764,-1450477.22,-1450477.22,0.00,RECONCILED,2026-08-22T07:03:24.116Z
ACC021808,366028.13,366028.13,0.00,RECONCILED,2026-08-22T07:03:24.116Z
ACC023596,673611.68,673611.68,0.00,RECONCILED,2026-08-22T07:03:24.116Z


In [0]:
(
    reconciliation_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold.balance_reconciliation"
    )
)

print("Balance reconciliation completed")

Balance reconciliation completed


##Reconciliation summary

In [0]:
display(
    spark.sql("""
        SELECT
            reconciliation_status,
            COUNT(*) AS account_count
        FROM workspace.gold.balance_reconciliation
        GROUP BY reconciliation_status
    """)
)

reconciliation_status,account_count
RECONCILED,135058
